In [78]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import tensorflow as tf

In [2]:
df = pd.read_csv("emergency_service.csv") # Data Loading
df.head()

,Timestamp,Incident_Severity,Incident_Type,Region_Type,Traffic_Congestion,Weather_Condition,Ambulance_Availability,Hospital_Capacity,Distance_to_Incident,Number_of_Injuries,Specialist_Availability,Road_Type,Emergency_Level,Ambulance_Speed,Fuel_Level,Weather_Impact,Dispatch_Coordinator,Response_Time
0,01/01/2018 00:00,Low,Cardiac Arrest,Suburban,High,Clear,Available,91,8.89,1,Unavailable,Highway,Major,43.549849,90.030756,Severe,AI,20.870220
1,01/01/2018 00:10,Low,Other,Urban,Moderate,Clear,Available,28,28.10,2,Unavailable,Highway,Critical,30.687975,88.255008,Moderate,Human,24.486195
2,01/01/2018 00:20,Medium,Cardiac Arrest,Suburban,High,Rainy,Available,83,40.43,1,Unavailable,Unpaved Road,Minor,44.456331,97.719622,No Impact,Human,18.760740
3,01/01/2018 00:30,Low,Accident,Urban,Moderate,Clear,Available,31,18.97,3,Available,Highway,Minor,35.879968,60.234672,No Impact,Human,18.169534
4,01/01/2018 00:40,Low,Cardiac Arrest,Urban,Moderate,Stormy,Available,52,16.31,3,Available,Highway,Minor,26.369383,71.083564,Moderate,Human,5.000000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 368065 entries, 0 to 368064
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Timestamp                368065 non-null  object 
 1   Incident_Severity        368065 non-null  object 
 2   Incident_Type            368065 non-null  object 
 3   Region_Type              368065 non-null  object 
 4   Traffic_Congestion       368065 non-null  object 
 5   Weather_Condition        368065 non-null  object 
 6   Ambulance_Availability   368065 non-null  object 
 7   Hospital_Capacity        368065 non-null  int64  
 8   Distance_to_Incident     368065 non-null  float64
 9   Number_of_Injuries       368065 non-null  int64  
 10  Specialist_Availability  368065 non-null  object 
 11  Road_Type                368065 non-null  object 
 12  Emergency_Level          368065 non-null  object 
 13  Ambulance_Speed          368065 non-null  float64
 14  Fuel

In [13]:
# Data Preprocessing
df_encoded = df.copy()
df_encoded.columns

Index(['Timestamp', 'Incident_Severity', 'Incident_Type', 'Region_Type',
       'Traffic_Congestion', 'Weather_Condition', 'Ambulance_Availability',
       'Hospital_Capacity', 'Distance_to_Incident', 'Number_of_Injuries',
       'Specialist_Availability', 'Road_Type', 'Emergency_Level',
       'Ambulance_Speed', 'Fuel_Level', 'Weather_Impact',
       'Dispatch_Coordinator', 'Response_Time'],
      dtype='object')

In [8]:
df_encoded.drop("Timestamp", axis=1, inplace=True)

In [9]:
df_encoded = pd.get_dummies(df_encoded, drop_first=True)

In [10]:
X = df_encoded.drop("Response_Time", axis=1)
y = df_encoded["Response_Time"]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Train-Test Split

In [17]:
# check that everything is ok
print(X.shape)
print(y.shape)

(368065, 25)
(368065,)


In [16]:
X.head()

,Hospital_Capacity,Distance_to_Incident,Number_of_Injuries,Ambulance_Speed,Fuel_Level,Incident_Severity_Low,Incident_Severity_Medium,Incident_Type_Cardiac Arrest,Incident_Type_Fire,Incident_Type_Other,...,Weather_Condition_Stormy,Ambulance_Availability_Unavailable,Specialist_Availability_Unavailable,Road_Type_Local Road,Road_Type_Unpaved Road,Emergency_Level_Major,Emergency_Level_Minor,Weather_Impact_No Impact,Weather_Impact_Severe,Dispatch_Coordinator_Human
0,91,8.89,1,43.549849,90.030756,True,False,True,False,False,...,False,False,True,False,False,True,False,False,True,False
1,28,28.10,2,30.687975,88.255008,True,False,False,False,True,...,False,False,True,False,False,False,False,False,False,True
2,83,40.43,1,44.456331,97.719622,False,True,True,False,False,...,False,False,True,False,True,False,True,True,False,True
3,31,18.97,3,35.879968,60.234672,True,False,False,False,False,...,False,False,False,False,False,False,True,True,False,True
4,52,16.31,3,26.369383,71.083564,True,False,True,False,False,...,True,False,False,False,False,False,True,False,False,True


In [37]:
selector = SelectKBest(score_func=f_regression, k=5)
selector.fit(X, y)

,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_r...001651D1A3EC0>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",5


In [38]:
feature_scores = pd.DataFrame({
    "Feature": X.columns,
    "Score": selector.scores_
})

feature_scores = feature_scores.sort_values(
    by="Score",
    ascending=False
)

feature_scores

,Feature,Score
17,Specialist_Availability_Unavailable,2.830874
15,Weather_Condition_Stormy,2.829592
14,Weather_Condition_Rainy,2.742275
19,Road_Type_Unpaved Road,2.515680
24,Dispatch_Coordinator_Human,2.059793
9,Incident_Type_Other,1.916161
7,Incident_Type_Cardiac Arrest,1.624567
20,Emergency_Level_Major,1.416418
4,Fuel_Level,1.110064
22,Weather_Impact_No Impact,1.020074


In [40]:
selected_features_kbest = X.columns[selector.get_support()]
print(selected_features_kbest)

Index(['Weather_Condition_Rainy', 'Weather_Condition_Stormy',
       'Specialist_Availability_Unavailable', 'Road_Type_Unpaved Road',
       'Dispatch_Coordinator_Human'],
      dtype='object')


In [22]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df)

                                Feature  Importance
3                       Ambulance_Speed    0.195385
4                            Fuel_Level    0.191985
1                  Distance_to_Incident    0.188385
0                     Hospital_Capacity    0.137868
2                    Number_of_Injuries    0.029897
18                 Road_Type_Local Road    0.017208
22             Weather_Impact_No Impact    0.016491
17  Specialist_Availability_Unavailable    0.016060
24           Dispatch_Coordinator_Human    0.015376
9                   Incident_Type_Other    0.014363
5                 Incident_Severity_Low    0.014283
7          Incident_Type_Cardiac Arrest    0.014192
12               Traffic_Congestion_Low    0.014099
13          Traffic_Congestion_Moderate    0.013756
21                Emergency_Level_Minor    0.013329
6              Incident_Severity_Medium    0.013220
14              Weather_Condition_Rainy    0.012909
11                    Region_Type_Urban    0.012771
16   Ambulan

In [24]:
selected_features = [
    "Ambulance_Speed",
    "Fuel_Level",
    "Distance_to_Incident",
    "Hospital_Capacity",
    "Number_of_Injuries"
]

X_selected = X[selected_features]

In [41]:
X_kbest = X[selected_features_kbest]

In [42]:
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(X_kbest, y, test_size=0.2, random_state=42)

In [45]:
lr_kbest = LinearRegression()
lr_kbest.fit(X_train_k, y_train_k)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [46]:
y_pred_kbest = lr_kbest.predict(X_test_k)

In [47]:
mae_kbest = mean_absolute_error(y_test_k, y_pred_kbest)

mse_kbest = mean_squared_error(y_test_k, y_pred_kbest)

rmse_kbest = mse_kbest ** 0.5

r2_kbest = r2_score(y_test_k, y_pred_kbest)

In [48]:
print("Linear Regression + SelectKBest")

print("MAE :", mae_kbest)
print("MSE :", mse_kbest)
print("RMSE:", rmse_kbest)
print("R²  :", r2_kbest)

Linear Regression + SelectKBest
MAE : 3.9446154049685345
MSE : 23.945892706541706
RMSE: 4.893454067071817
R²  : -6.868381949010072e-05


In [25]:
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

In [58]:
lr = LinearRegression()
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [59]:
y_pred_lr = lr.predict(X_test)

In [60]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = mse_lr ** 0.5
r2_lr = r2_score(y_test, y_pred_lr)

In [61]:
print("Linear Regression Results")
print("MAE :", mae_lr)
print("MSE :", mse_lr)
print("RMSE:", rmse_lr)
print("R²  :", r2_lr)

Linear Regression Results
MAE : 3.9445931579514824
MSE : 23.94666236320285
RMSE: 4.893532707891391
R²  : -0.00010082751668027967


In [62]:
dt = DecisionTreeRegressor(
    random_state=42
)

In [63]:
dt.fit(X_train_k, y_train_k)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_l

In [64]:
y_pred_dt = dt.predict(X_test_k)

In [65]:
mae_dt = mean_absolute_error(y_test_k, y_pred_dt)
mse_dt = mean_squared_error(y_test_k, y_pred_dt)
rmse_dt = mse_dt ** 0.5
r2_dt = r2_score(y_test_k, y_pred_dt)

In [66]:
print("Decision Tree Results")
print("MAE :", mae_dt)
print("MSE :", mse_dt)
print("RMSE:", rmse_dt)
print("R²  :", r2_dt)

Decision Tree Results
MAE : 3.944861462466285
MSE : 23.948409443462303
RMSE: 4.89371121373772
R²  : -0.00017379202366307744


In [69]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_k)
X_test_scaled = scaler.transform(X_test_k)

In [70]:
knn = KNeighborsRegressor(
    n_neighbors=5
)
knn.fit(X_train_scaled, y_train_k)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [71]:
y_pred_knn = knn.predict(X_test_scaled)

In [72]:
mae_knn = mean_absolute_error(y_test_k, y_pred_knn)

mse_knn = mean_squared_error(y_test_k, y_pred_knn)

rmse_knn = mse_knn ** 0.5

r2_knn = r2_score(y_test_k, y_pred_knn)

In [73]:
print("KNN Results")
print("MAE :", mae_knn)
print("MSE :", mse_knn)
print("RMSE:", rmse_knn)
print("R²  :", r2_knn)

KNN Results
MAE : 4.166898975696632
MSE : 26.794307153165555
RMSE: 5.17632177836401
R²  : -0.11902896320925493


In [79]:
tf.random.set_seed(42)
model = Sequential([
    Input(shape=(5,)),
    Dense(10, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse'
)

In [80]:
history = model.fit(
    X_train_scaled,
    y_train_k,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - loss: 38.5226 - val_loss: 24.0825
Epoch 2/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 23.9362 - val_loss: 24.0571
Epoch 3/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 25s 3ms/step - loss: 23.9214 - val_loss: 24.0522
Epoch 4/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 40s 3ms/step - loss: 23.9168 - val_loss: 24.0482
Epoch 5/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 38s 3ms/step - loss: 23.9136 - val_loss: 24.0462
Epoch 6/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - loss: 23.9113 - val_loss: 24.0444
Epoch 7/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - loss: 23.9101 - val_loss: 24.0433
Epoch 8/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - loss: 23.9090 - val_loss: 24.0431
Epoch 9/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 23.9082 - val_loss: 24.0420
Epoch 10/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - loss: 23.9075 - val_loss: 24.0413
Epoch 11/50
7362/7362 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 23.9070 - val_loss: 24.04

In [81]:
y_pred_ann = model.predict(X_test_scaled)
y_pred_ann = y_pred_ann.flatten()

2301/2301 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step


In [82]:
mae_ann = mean_absolute_error(y_test_k, y_pred_ann)
mse_ann = mean_squared_error(y_test_k, y_pred_ann)
rmse_ann = mse_ann ** 0.5
r2_ann = r2_score(y_test_k, y_pred_ann)

In [83]:
print("ANN Results")
print("MAE :", mae_ann)
print("MSE :", mse_ann)
print("RMSE:", rmse_ann)
print("R²  :", r2_ann)

ANN Results
MAE : 3.944477027723002
MSE : 23.951185285584373
RMSE: 4.893994818712457
R²  : -0.0002897214154600203


In [84]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "KNN",
        "ANN"
    ],
    "MAE": [
        mae_kbest,
        mae_dt,
        mae_knn,
        mae_ann
    ],
    "RMSE": [
        rmse_kbest,
        rmse_dt,
        rmse_knn,
        rmse_ann
    ],
    "R2": [
        r2_kbest,
        r2_dt,
        r2_knn,
        r2_ann
    ]
})

comparison

,Model,MAE,RMSE,R2
0,Linear Regression,3.944615,4.893454,-0.000069
1,Decision Tree,3.944861,4.893711,-0.000174
2,KNN,4.166899,5.176322,-0.119029
3,ANN,3.944477,4.893995,-0.000290


In [85]:
#Four predictive models were evaluated using selected features.
#Linear Regression achieved the best overall performance,
#although all models produced low predictive accuracy due to the weak relationships between the available features and response time.